# Light Training Notebook (Container MLflow)

Use this notebook for quick experiments with different hyperparameters while logging runs to your MLflow container by default.

## 1) Start MLflow container

From project root:

```powershell
podman compose up -d mlflow
```

Tracking UI: http://localhost:5000

In [1]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path

import mlflow
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms.functional import to_tensor
from tqdm.auto import tqdm

from model import build_model
from PIL import Image


e:\Academics\sem_6\ML\prac 6\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Quick experiment config (change these per run)
PROJECT_ROOT = Path('.').resolve()
DATA_ROOT = PROJECT_ROOT / 'fruit_detection'
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'test'

CLASS_TO_ID = {'apple': 1, 'banana': 2, 'orange': 3}
NUM_CLASSES = 4

IMG_SIZE = 512
BATCH_SIZE = 8
VAL_BATCH_SIZE = 8
EPOCHS = 50
FREEZE_BACKBONE_EPOCHS = 10
BASE_LR = 0.005
FINETUNE_LR = 0.001
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
NUM_WORKERS = 0
PIN_MEMORY = True
USE_AMP = True
RUN_NAME = 'light-fasterrcnn-v3'

TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000')
EXPERIMENT_NAME = os.environ.get('MLFLOW_EXPERIMENT_NAME', 'fruit-detection-light')

OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



In [7]:
class FruitVOCDataset(Dataset):
    def __init__(self, image_dir: Path, class_to_id: dict[str, int], img_size: int):
        self.image_dir = image_dir
        self.class_to_id = class_to_id
        self.img_size = img_size
        self.images = sorted(image_dir.glob('*.jpg'))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        xml_path = img_path.with_suffix('.xml')

        image_pil = Image.open(img_path).convert('RGB')
        orig_w, orig_h = image_pil.size
        image_pil = image_pil.resize((self.img_size, self.img_size), Image.BILINEAR)
        image = to_tensor(image_pil)

        sx = self.img_size / orig_w
        sy = self.img_size / orig_h

        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []
        for obj in root.findall('object'):
            name = obj.find('name').text.strip().lower()
            if name not in self.class_to_id:
                continue
            bnd = obj.find('bndbox')
            xmin = float(bnd.find('xmin').text) * sx
            ymin = float(bnd.find('ymin').text) * sy
            xmax = float(bnd.find('xmax').text) * sx
            ymax = float(bnd.find('ymax').text) * sy
            if xmax <= xmin or ymax <= ymin:
                continue
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_id[name])

        if boxes:
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)

        target = {
            'boxes': boxes_t,
            'labels': labels_t,
            'image_id': torch.tensor([idx], dtype=torch.int64),
        }
        return image, target


def collate_fn(batch):
    return [b[0] for b in batch], [b[1] for b in batch]



In [8]:
def set_backbone_trainability(model, freeze_backbone: bool):
    for p in model.backbone.parameters():
        p.requires_grad = not freeze_backbone

    # Keep a small fine-tuning footprint when backbone is unfrozen.
    if not freeze_backbone:
        for p in model.backbone.body.layer1.parameters():
            p.requires_grad = False
        for p in model.backbone.body.layer2.parameters():
            p.requires_grad = False
        for p in model.backbone.body.layer3.parameters():
            p.requires_grad = False


def mean_detection_loss(model, loader, device, use_amp=False):
    was_training = model.training
    model.train()
    total = 0.0
    count = 0

    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            with torch.autocast(device_type='cuda', enabled=use_amp):
                loss_dict = model(images, targets)
                loss_value = sum(loss for loss in loss_dict.values()).item()
            total += loss_value
            count += 1

    if not was_training:
        model.eval()

    return total / max(count, 1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp_enabled = bool(USE_AMP and device.type == 'cuda')

train_ds = FruitVOCDataset(TRAIN_DIR, CLASS_TO_ID, IMG_SIZE)
val_ds = FruitVOCDataset(VAL_DIR, CLASS_TO_ID, IMG_SIZE)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_ds,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

model = build_model(NUM_CLASSES).to(device)
set_backbone_trainability(model, freeze_backbone=FREEZE_BACKBONE_EPOCHS > 0)

def make_optimizer(lr):
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.SGD(params, lr=lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

optimizer = make_optimizer(BASE_LR)
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment:', EXPERIMENT_NAME)
print('Device:', device)
print('AMP enabled:', amp_enabled)
print('Train images:', len(train_ds), 'Val images:', len(val_ds))



Tracking URI: http://localhost:5000
Experiment: fruit-detection-light
Device: cuda
AMP enabled: True
Train images: 240 Val images: 60


C:\Users\veryb\AppData\Local\Temp\ipykernel_11508\1669358514.py:68: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)


In [9]:
with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.set_tags({
        'project': 'fruit-detection',
        'notebook': 'practice_7_light.ipynb',
        'tracking_target': 'container-mlflow'
    })

    mlflow.log_params({
        'num_classes': NUM_CLASSES,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'val_batch_size': VAL_BATCH_SIZE,
        'epochs': EPOCHS,
        'freeze_backbone_epochs': FREEZE_BACKBONE_EPOCHS,
        'base_lr': BASE_LR,
        'finetune_lr': FINETUNE_LR,
        'momentum': MOMENTUM,
        'weight_decay': WEIGHT_DECAY,
        'use_amp': amp_enabled,
        'train_images': len(train_ds),
        'val_images': len(val_ds),
    })

    current_lr = BASE_LR
    for epoch in range(1, EPOCHS + 1):
        if epoch == FREEZE_BACKBONE_EPOCHS + 1 and FREEZE_BACKBONE_EPOCHS > 0:
            set_backbone_trainability(model, freeze_backbone=False)
            current_lr = FINETUNE_LR
            optimizer = make_optimizer(current_lr)

        model.train()
        running = 0.0
        batches = 0

        progress = tqdm(
            train_loader,
            total=len(train_loader),
            desc=f'Epoch {epoch}/{EPOCHS}',
            leave=False,
            dynamic_ncols=True
        )

        for images, targets in progress:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=amp_enabled):
                loss_dict = model(images, targets)
                loss = sum(loss for loss in loss_dict.values())

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running += loss.item()
            batches += 1
            progress.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{current_lr:.4g}'})

        train_loss = running / max(batches, 1)
        val_loss = mean_detection_loss(model, val_loader, device, use_amp=amp_enabled)

        mlflow.log_metrics(
            {'train_loss': train_loss, 'val_loss': val_loss, 'lr': current_lr},
            step=epoch,
        )
        print(f'Epoch {epoch}/{EPOCHS} - train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}, lr: {current_lr:.4g}')

    state_path = OUTPUT_DIR / 'fasterrcnn_state_dict.pth'
    torch.save(model.state_dict(), state_path)

    model.eval()
    scripted = torch.jit.script(model.cpu())
    script_path = OUTPUT_DIR / 'fruits_model_light.pt'
    scripted.save(str(script_path))

    mlflow.log_artifact(str(state_path), artifact_path='checkpoints')
    mlflow.log_artifact(str(script_path), artifact_path='torchscript')

    print('Run ID:', run.info.run_id)
    print('Artifacts:', OUTPUT_DIR)



Epoch 1/50 - train_loss: 0.5499, val_loss: 0.3132, lr: 0.005


Epoch 2/50 - train_loss: 0.2527, val_loss: 0.2162, lr: 0.005


Epoch 3/50 - train_loss: 0.2031, val_loss: 0.2016, lr: 0.005


Epoch 4/50 - train_loss: 0.1899, val_loss: 0.1923, lr: 0.005


Epoch 5/50 - train_loss: 0.1759, val_loss: 0.1786, lr: 0.005


...

Epoch 46/50 - train_loss: 0.0687, val_loss: 0.1587, lr: 0.001


Epoch 47/50 - train_loss: 0.0681, val_loss: 0.1576, lr: 0.001


Epoch 48/50 - train_loss: 0.0671, val_loss: 0.1554, lr: 0.001


Epoch 49/50 - train_loss: 0.0662, val_loss: 0.1588, lr: 0.001


Epoch 50/50 - train_loss: 0.0645, val_loss: 0.1614, lr: 0.001
Run ID: 2ddf441dce2a407c86b3a10fadb62a4b
Artifacts: E:\Academics\sem_6\ML\prac 6\artifacts\light-fasterrcnn-v3
🏃 View run light-fasterrcnn-v3 at: http://localhost:5000/#/experiments/2/runs/2ddf441dce2a407c86b3a10fadb62a4b
🧪 View experiment at: http://localhost:5000/#/experiments/2


## 2) Verify in UI

Open http://localhost:5000 and check experiment `fruit-detection-light`.

If needed, override target server before launching Jupyter:

```powershell
$env:MLFLOW_TRACKING_URI = 'http://localhost:5000'
$env:MLFLOW_EXPERIMENT_NAME = 'fruit-detection-light'
jupyter lab
```